In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


# *Step 1: Import Python libraries*

In [3]:
%%time
### cell 0 ###

import os
import glob
import numpy as np 
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 5000)

CPU times: user 120 ms, sys: 7.99 ms, total: 128 ms
Wall time: 126 ms


In [4]:
import time
start_time = time.time()

# *Step 2: Define helper functions*

In [5]:
%%time
### cell 1 ###

def bar_chart_multiple_choice(response_counts,title,y_axis_title,orientation='h',num_choices=15): 
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'+title+'.csv',index=True)
    # STEFANOS: It was in the plotting arguments. I kept the computations
    tmp_cmp = max(response_counts_series[0:num_choices]*1.2)    
    
def bar_chart_multiple_choice_multiple_selection(df,title,orientation='v'):  
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts*100/len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'+title+'.csv',index=True)

def bar_chart_multiple_years(df,x_column,y_column,title,y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'+title+'.csv',index=True)

def create_choropleth_map(df, column, title, max_value):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'+title+'.csv',index=True)

CPU times: user 5 μs, sys: 1 μs, total: 6 μs
Wall time: 8.82 μs


In [6]:
%%time
### cell 2 ###

def grab_subset_of_data(original_df, question_of_interest):
    # adapted from https://www.kaggle.com/siddhantsadangi/your-country-vs-the-world-24-factors-wip
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns,mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def create_dataframe_of_counts(dataframe,column,rename_index,rename_column,return_percentages=False): 
    df = dataframe[column].value_counts().reset_index() 
    if return_percentages==True:
        df[column] = (df[column]*100)/(df[column].sum())
    df = pd.DataFrame(df) 
    df = df.rename({'index':rename_index, column:rename_column}, axis='columns')
    return df

def count_then_return_percent(dataframe,x_axis_title): 
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts*100/(dataframe[x_axis_title].count()),1)
    return percentages

def add_year_column_to_dataframes(df_2022,df_2021,df_2020,df_2019,df_2018,df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return df_2017,df_2018,df_2019,df_2020,df_2021,df_2022
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return df_2018,df_2019,df_2020,df_2021,df_2022

def convert_df_of_counts_to_percentages(df,df_counts): 
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1]*100/df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2]*100/df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3]*100/df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4]*100/df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5]*100/df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018','2019','2020','2021','2022']
    return df_combined_percentages

CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 6.91 μs


In [7]:
%%time
### cell 3 ###

def combine_subset_of_data_from_multiple_years(question_of_interest,x_axis_title,include_2017=None): 
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent(responses_df_2022,question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent(responses_df_2021,question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent(responses_df_2020,question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent(responses_df_2019,question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent(responses_df_2018,question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent(responses_df_2017,question_of_interest).sort_index())

        add_year_column_to_dataframes(
            df_2022,
            df_2021,
            df_2020,
            df_2019,
            df_2018,
            df_2017)

        df_2022.columns = ['percentage','year']
        df_2021.columns = ['percentage','year']
        df_2020.columns = ['percentage','year']
        df_2019.columns = ['percentage','year']
        df_2018.columns = ['percentage','year']
        df_2017.columns = ['percentage','year']
        
        df_combined = pd.concat(
            [df_2017,
            df_2018,
            df_2019,
            df_2020,
            df_2021,
            df_2022])

        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage','year',x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent(responses_df_2022,question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent(responses_df_2021,question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent(responses_df_2020,question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent(responses_df_2019,question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent(responses_df_2018,question_of_interest).sort_index())

        add_year_column_to_dataframes(
            df_2022,
            df_2021,
            df_2020,
            df_2019,
            df_2018)

        df_2022.columns = ['percentage','year']
        df_2021.columns = ['percentage','year']
        df_2020.columns = ['percentage','year']
        df_2019.columns = ['percentage','year']
        df_2018.columns = ['percentage','year']

        df_combined = pd.concat(
            [df_2018,
            df_2019,
            df_2020,
            df_2021,
            df_2022])

        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage','year',x_axis_title]
        return df_combined        
    
    

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions(question_of_interest,include_2017=None): 
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data(responses_df_2018, question_of_interest)

        add_year_column_to_dataframes(
            df_2022,
            df_2021,
            df_2020,
            df_2019,
            df_2018)

        df_combined = pd.concat(
            [df_2018,
            df_2019,
            df_2020,
            df_2021,
            df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()

        return df_combined, df_combined_counts
    
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data(responses_df_2017, question_of_interest)

        add_year_column_to_dataframes(
            df_2022,
            df_2021,
            df_2020,
            df_2019,
            df_2018,
            df_2017)

        df_combined = pd.concat(
            [df_2017,
            df_2018,
            df_2019,
            df_2020,
            df_2021,
            df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()

        return df_combined, df_combined_counts

def load_survey_data(base_dir,file_name,rows_to_skip=1):
    file_path = os.path.join(base_dir,file_name)
    df = pd.read_csv(file_path,low_memory=False,encoding='ISO-8859-1',skiprows=rows_to_skip)
    file_name = 'kaggle_survey_'+base_dir[-5:-1]+'.csv'
    files_present = glob.glob(file_name)
    if not files_present:
        df.to_csv(file_name,index=False)    
    return df

CPU times: user 3 μs, sys: 1e+03 ns, total: 4 μs
Wall time: 6.44 μs


# *Step 3: Load the data* 
6 datasets for 6 years (2017-2022)

In [8]:
%%time
### cell 4 ###

def load_survey_data_1(base_dir,file_name,rows_to_skip=1):
    file_path = os.path.join(base_dir,file_name)
    df = pd.read_csv(file_path,low_memory=False,encoding='ISO-8859-1',skiprows=rows_to_skip)
    file_name = 'kaggle_survey_'+base_dir[-5:-1]+'.csv'
    files_present = glob.glob(file_name)
    if not files_present:
        df.to_csv(file_name,index=False)    
    return df

directory = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/'
if not os.path.exists(directory):
    os.makedirs(directory, exist_ok=True)

    
directory = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'
if not os.path.exists(directory):
    os.makedirs(directory, exist_ok=True)


directory = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/charts/'
if not os.path.exists(directory):
    os.makedirs(directory, exist_ok=True)


base_dir_2017 = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2017/'
file_name_2017 = 'multipleChoiceResponses.csv'
responses_df_2017 = load_survey_data_1(base_dir_2017,file_name_2017,rows_to_skip=0)

base_dir_2018 = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2018/'
file_name_2018 = 'multipleChoiceResponses.csv'
responses_df_2018 = load_survey_data_1(base_dir_2018,file_name_2018)

base_dir_2019 = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2019/'
file_name_2019 = 'multiple_choice_responses.csv'
responses_df_2019 = load_survey_data_1(base_dir_2019,file_name_2019)

base_dir_2020 = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2020/'
file_name_2020 = 'kaggle_survey_2020_responses.csv'
responses_df_2020 = load_survey_data_1(base_dir_2020,file_name_2020)

base_dir_2021 = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2021/'
file_name_2021 = 'kaggle_survey_2021_responses.csv'
responses_df_2021 = load_survey_data_1(base_dir_2021,file_name_2021)

base_dir_2022 = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2022/'
file_name_2022 = 'kaggle_survey_2022_responses.csv'
responses_df_2022 = load_survey_data_1(base_dir_2022,file_name_2022)

CPU times: user 3.72 s, sys: 810 ms, total: 4.53 s
Wall time: 4.53 s


# *Step 4: Clean the data and apply filters*
Filter according to job title, country of residence, etc

In [9]:
%%time
### cell 5 ###

# Drop columns containing '- Text' using GPU-accelerated string methods
responses_df_2018 = responses_df_2018.loc[:, ~responses_df_2018.columns.str.contains('- Text')]
responses_df_2019 = responses_df_2019.loc[:, ~responses_df_2019.columns.str.contains('- Text')]

# Chain all of the 2022 column-name replacements and assign once
cols = responses_df_2022.columns
cols = cols.str.replace('Scikit-learn', 'Scikit—learn')
cols = cols.str.replace('peer-reviewed', 'peer—reviewed')
cols = cols.str.replace('Cloud-certification', 'Cloud—certification')
cols = cols.str.replace('U-Net, Mask R-CNN', 'U—Net, Mask R—CNN')
cols = cols.str.replace('Encoder-decoder', 'Encoder—decoder')
cols = cols.str.replace('GPT-3', 'GPT—3')
cols = cols.str.replace('gpt-3', 'gpt—3')
cols = cols.str.replace('pre-trained', 'pre—trained')
cols = cols.str.replace('What-if', 'What—if')
cols = cols.str.replace('Audit-AI', 'Audit—AI')
responses_df_2022.columns = cols

CPU times: user 871 ms, sys: 66 ms, total: 937 ms
Wall time: 906 ms


In [10]:
# # # Filter data according to whether or not the respondent gave specific answers (if you so decide)

# # # e.g.
# job_title = 'Data Scientist'
# # country = 'India'

# question_of_interest = 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
# question_of_interest_2017 = 'CurrentJobTitleSelect'
# responses_df_2017 = responses_df_2017[responses_df_2017[question_of_interest_2017]==job_title] 
# responses_df_2018 = responses_df_2018[responses_df_2018[question_of_interest]==job_title] 
# responses_df_2019 = responses_df_2019[responses_df_2019[question_of_interest]==job_title] 
# responses_df_2020 = responses_df_2020[responses_df_2020[question_of_interest]==job_title] 
# responses_df_2021 = responses_df_2021[responses_df_2021[question_of_interest]==job_title] 
# responses_df_2022 = responses_df_2022[responses_df_2022[question_of_interest]==job_title] 

# # question_of_interest = 'In which country do you currently reside?'
# # question_of_interest_2017 = 'Country'
# # responses_df_2017 = responses_df_2017[responses_df_2017[question_of_interest_2017]==country] 
# # responses_df_2018 = responses_df_2018[responses_df_2018[question_of_interest]==country] 
# # responses_df_2019 = responses_df_2019[responses_df_2019[question_of_interest]==country] 
# # responses_df_2020 = responses_df_2020[responses_df_2020[question_of_interest]==country] 
# # responses_df_2021 = responses_df_2021[responses_df_2021[question_of_interest]==country] 
# # responses_df_2022 = responses_df_2022[responses_df_2022[question_of_interest]==country] 

# *Step 5: Preview the data*

`kaggle_survey_2022_responses.csv`: **43 questions and 23,997 responses**

- Responses to multiple choice questions (only a single choice can be selected)
were recorded in individual columns. Responses to multiple selection questions
(multiple choices can be selected) were split into multiple columns (with one
column per answer choice).

`kaggle_survey_2022_answer_choices.pdf`: **list of answer choices for every question**

- With footnotes describing which questions were asked to which respondents.

`kaggle_survey_2022_methodology.pdf`: **a description of how the survey was conducted**

- You can ask additional questions by posting in the pinned Q&A thread.


In [11]:
%%time
### cell 6 ###

# Preview the data
responses_df_2022.head(5)

CPU times: user 28.9 ms, sys: 4.14 ms, total: 33.1 ms
Wall time: 30.8 ms


,Duration (in seconds),What is your age (# years)?,What is your gender? - Selected Choice,In which country do you currently reside?,"Are you currently a student? (high school, university, or graduate)",On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Coursera,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - edX,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Kaggle Learn Courses,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - DataCamp,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Fast.ai,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Udacity,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Udemy,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - LinkedIn Learning,"On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Cloud—certification programs (direct from AWS, Azure, GCP, or similar)",On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - University Courses (resulting in a university degree),On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - None,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Other,What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - University courses,"What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Online courses (Coursera, EdX, etc)","What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Social media platforms (Reddit, Twitter, etc)","What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Video platforms (YouTube, Twitch, etc)","What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Kaggle (notebooks, competitions, etc)",What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - None / I do not study data science,What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Other,What is the highest level of formal education that you have attained or plan to attain within the next 2 years?,"Have you ever published any academic research (papers, preprints, conference proceedings, etc)?","Did your research make use of machine learning? - Yes, the research made advances related to some novel machine learning method (theoretical research)","Did your research make use of machine learning? - Yes, the research made use of machine learning as a tool (applied research)",Did your research make use of machine learning? - No,For how many years have you been writing code and/or programming?,What programming languages do you use on a regular basis? (Select all that apply) - Selected Choice - Python,What programming languages do you use on a regular basis? (Select all that apply) - Selected Choice - R,What programming languages do you use on a regular basis? (Select all that apply) - Selected Choice - SQL,What programming langu

# *Step 6: Create data visualizations*

In [12]:
%%time
### cell 7 ###

def create_choropleth_map_16(df, column, title, max_value):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def create_dataframe_of_counts_16(dataframe, column, rename_index, rename_column, return_percentages=False):
    df = dataframe[column].value_counts().reset_index()
    if return_percentages == True:
        df[column] = df[column] * 100 / df[column].sum()
    df = pd.DataFrame(df)
    df = df.rename({'index': rename_index, column: rename_column}, axis='columns')
    return df

responses_per_country_df = create_dataframe_of_counts_16(responses_df_2022[1:], 'In which country do you currently reside?', 'country', '# of respondents', return_percentages=False)
percentages_per_country_df = create_dataframe_of_counts_16(responses_df_2022[1:], 'In which country do you currently reside?', 'country', '% of respondents', return_percentages=False)
title_for_chart = 'Percentage of total responses for most common countries in 2022'
label_for_legend = '% of respondents'
create_choropleth_map_16(percentages_per_country_df, label_for_legend, title_for_chart, max_value=5)
print('Note that countries with less than 50 responses were replaced with the country name "other" (which does not show up on this map)')

Note that countries with less than 50 responses were replaced with the country name "other" (which does not show up on this map)
CPU times: user 194 ms, sys: 21.5 ms, total: 216 ms
Wall time: 203 ms


In [13]:
%%time
### cell 8 ###

def bar_chart_multiple_choice_17(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_17(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'In which country do you currently reside?'
responses_df_2022.rename(columns={'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom (UK)'}, inplace=True)
responses_df_2022.replace(['United Kingdom of Great Britain and Northern Ireland'], 'United Kingdom (UK)', inplace=True)
responses_in_order = ['Other', 'India', 'United States of America', 'Brazil', 'Nigeria', 'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico', 'Turkey', 'Russia']
responses_df_2022[question_of_interest][~responses_df_2022[question_of_interest].isin(responses_in_order)] = 'Other'
percentages = count_then_return_percent_17(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
percentages = percentages.sort_values(ascending=True)
title_for_chart = 'Most Common Nationalities Kaggle'
title_for_y_axis = '% of respondents'
orientation_for_chart = 'h'
bar_chart_multiple_choice_17(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=25)
print("'Other' = any country not shown")

'Other' = any country not shown
CPU times: user 571 ms, sys: 43.6 ms, total: 615 ms
Wall time: 598 ms


In [14]:
%%time
### cell 9 ###

def bar_chart_multiple_years_18(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_18(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_18(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_18(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_18(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_18(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_18(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_18(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_18(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_18(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_18(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_18(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_18(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_18(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_18(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_18(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_18(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

question_name_alternate = 'Country'
responses_df_2022.rename(columns={'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'}, inplace=True)
responses_df_2022.replace(['United Kingdom (UK)'], 'United Kingdom of Great Britain and Northern Ireland', inplace=True)
responses_df_2017[question_name_alternate].replace(['United States'], 'United States of America', inplace=True)
responses_df_2017[question_name_alternate].replace(["People 's Republic of China"], 'China', inplace=True)
responses_df_2017[question_name_alternate].replace(['United Kingdom'], 'United Kingdom of Great Britain and Northern Ireland', inplace=True)
responses_df_2017.rename(columns={question_name_alternate: 'In which country do you currently reside?'}, inplace=True)
question_name_alternate = 'CurrentJobTitleSelect'
responses_df_2017.rename(columns={question_name_alternate: 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'}, inplace=True)
subset_of_countries = ['Other', 'India', 'United States of America', 'Brazil', 'Nigeria', 'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico', 'Turkey', 'Russia']
question_name = 'In which country do you currently reside?'
responses_df_2017[question_name][~responses_df_2017[question_name].isin(subset_of_countries)] = 'Other'
responses_df_2018[question_name][~responses_df_2018[question_name].isin(subset_of_countries)] = 'Other'
responses_df_2019[question_name][~responses_df_2019[question_name].isin(subset_of_countries)] = 'Other'
responses_df_2020[question_name][~responses_df_2020[question_name].isin(subset_of_countries)] = 'Other'
responses_df_2021[question_name][~responses_df_2021[question_name].isin(subset_of_countries)] = 'Other'
responses_df_2022[question_name][~responses_df_2022[question_name].isin(subset_of_countries)] = 'Other'
question_of_interest = 'In which country do you currently reside?'
column_of_interest = 'percentage'
title_for_chart = 'Most Common Nationalities on Kaggle (2017-2022)'
title_for_x_axis = ''
title_for_y_axis = '% of respondents'
country_df_combined = combine_subset_of_data_from_multiple_years_18(question_of_interest, title_for_x_axis, include_2017='yes')
country_df_combined = country_df_combined.sort_values(by=['year', 'percentage'], ascending=True)
country_df_combined.rename(columns={'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom (UK)'}, inplace=True)
country_df_combined.replace(['United Kingdom of Great Britain and Northern Ireland'], 'United Kingdom', inplace=True)
bar_chart_multiple_years_18(country_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)
print("'Other' = any country not shown")

'Other' = any country not shown
CPU times: user 1.18 s, sys: 150 ms, total: 1.33 s
Wall time: 1.28 s


In [15]:
%%time
### cell 10 ###

def bar_chart_multiple_choice_19(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_19(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'What is your age (# years)?'
percentages = count_then_return_percent_19(responses_df_2022, question_of_interest).sort_index()
title_for_chart = 'Age Distributions on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_19(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 93.6 ms, sys: 4.75 ms, total: 98.3 ms
Wall time: 90.8 ms


In [16]:
%%time
### cell 11 ###

def bar_chart_multiple_years_20(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_20(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_20(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_20(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_20(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_20(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_20(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_20(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_20(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_20(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_20(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_20(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_20(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_20(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_20(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_20(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_20(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

question_of_interest = 'What is your age (# years)?'
column_of_interest = 'percentage'
title_for_chart = 'Age distributions on Kaggle (2018-2022)'
title_for_x_axis = ''
title_for_y_axis = '% of respondents'
responses_df_2018[question_of_interest].replace(['70-79', '80+'], '70+', inplace=True)
age_df_combined = combine_subset_of_data_from_multiple_years_20(question_of_interest, title_for_x_axis)
bar_chart_multiple_years_20(age_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 454 ms, sys: 28.1 ms, total: 482 ms
Wall time: 456 ms


In [17]:
%%time
### cell 12 ###

def bar_chart_multiple_choice_21(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_21(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'What is your gender? - Selected Choice'
responses_in_order = ['Man', 'Woman', 'Nonbinary', 'Prefer to self-describe', 'Prefer not to say']
percentages = count_then_return_percent_21(responses_df_2022, question_of_interest).sort_index()[responses_in_order].iloc[::-1]
title_for_chart = 'Gender Distributions on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_21(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 108 ms, sys: 9.16 ms, total: 118 ms
Wall time: 107 ms


In [18]:
%%time
### cell 13 ###

def bar_chart_multiple_years_22(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_22(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_22(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_22(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_22(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_22(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_22(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_22(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_22(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_22(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_22(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_22(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_22(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_22(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_22(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_22(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_22(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

responses_df_2017['GenderSelect'].replace(['Male'], 'Man', inplace=True)
responses_df_2017['GenderSelect'].replace(['Female'], 'Woman', inplace=True)
responses_df_2017['GenderSelect'].replace(['A different identity', 'Non-binary, genderqueer, or gender non-conforming'], 'Prefer to self-describe', inplace=True)
responses_df_2017['GenderSelect'].replace(['Non-binary, genderqueer, or gender non-conforming'], 'Nonbinary', inplace=True)
responses_df_2018['What is your gender? - Selected Choice'].replace(['Male'], 'Man', inplace=True)
responses_df_2018['What is your gender? - Selected Choice'].replace(['Female'], 'Woman', inplace=True)
responses_df_2019['What is your gender? - Selected Choice'].replace(['Male'], 'Man', inplace=True)
responses_df_2019['What is your gender? - Selected Choice'].replace(['Female'], 'Woman', inplace=True)
responses_df_2017['GenderSelect'].replace(['Nonbinary', 'Prefer not to say'], 'Prefer to self-describe', inplace=True)
responses_df_2017.columns = responses_df_2017.columns.str.replace('GenderSelect', 'What is your gender? - Selected Choice', regex=False)
responses_df_2018['What is your gender? - Selected Choice'].replace(['Nonbinary', 'Prefer not to say'], 'Prefer to self-describe', inplace=True)
responses_df_2019['What is your gender? - Selected Choice'].replace(['Nonbinary', 'Prefer not to say'], 'Prefer to self-describe', inplace=True)
responses_df_2020['What is your gender? - Selected Choice'].replace(['Nonbinary', 'Prefer not to say'], 'Prefer to self-describe', inplace=True)
responses_df_2021['What is your gender? - Selected Choice'].replace(['Nonbinary', 'Prefer not to say'], 'Prefer to self-describe', inplace=True)
responses_df_2022['What is your gender? - Selected Choice'].replace(['Nonbinary', 'Prefer not to say'], 'Prefer to self-describe', inplace=True)
question_of_interest = 'What is your gender? - Selected Choice'
column_of_interest = 'percentage'
title_for_chart = 'Gender distributions on Kaggle (2018-2022)'
title_for_x_axis = ''
title_for_y_axis = '% of respondents'
age_df_combined = combine_subset_of_data_from_multiple_years_22(question_of_interest, title_for_x_axis, include_2017='yes')
age_df_combined = age_df_combined.sort_values(by=['year', 'percentage'], ascending=True)
bar_chart_multiple_years_22(age_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 768 ms, sys: 42.7 ms, total: 810 ms
Wall time: 764 ms


In [19]:
%%time
### cell 14 ###

def bar_chart_multiple_choice_23(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_23(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Are you currently a student? (high school, university, or graduate)'
percentages = count_then_return_percent_23(responses_df_2022, question_of_interest).sort_index()
title_for_chart = 'Students status for Kaggle Survey participants'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_23(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 97.3 ms, sys: 37 μs, total: 97.4 ms
Wall time: 89.9 ms


In [20]:
%%time
### cell 15 ###

def bar_chart_multiple_choice_24(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_24(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

USA_responses_df_2022 = responses_df_2022[responses_df_2022['In which country do you currently reside?'] == 'United States of America']
question_of_interest = 'Are you currently a student? (high school, university, or graduate)'
percentages = count_then_return_percent_24(USA_responses_df_2022, question_of_interest).sort_index()
title_for_chart = 'Students status for Kaggle Survey participants from the USA'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_24(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)
India_responses_df_2022 = responses_df_2022[responses_df_2022['In which country do you currently reside?'] == 'India']
question_of_interest = 'Are you currently a student? (high school, university, or graduate)'
percentages = count_then_return_percent_24(India_responses_df_2022, question_of_interest).sort_index()
title_for_chart = 'Students status for Kaggle Survey participants from India'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_24(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 248 ms, sys: 35.4 ms, total: 283 ms
Wall time: 267 ms


In [21]:
%%time
### cell 16 ###

def bar_chart_multiple_choice_multiple_selection_25(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_25(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'On which platforms have you begun or completed data science courses?'
title_for_chart = 'Most popular online educational platforms for data science'
online_learning_platforms_df_2022 = grab_subset_of_data_25(responses_df_2022, question_of_interest)
online_learning_platforms_df_2022.columns = online_learning_platforms_df_2022.columns.str.replace('(direct from AWS, Azure, GCP, or similar)', '', regex=False)
online_learning_platforms_df_2022.columns = online_learning_platforms_df_2022.columns.str.replace('(resulting in a university degree)', '', regex=False)
bar_chart_multiple_choice_multiple_selection_25(online_learning_platforms_df_2022, title_for_chart, orientation='h')

CPU times: user 208 ms, sys: 0 ns, total: 208 ms
Wall time: 200 ms


In [22]:
%%time
### cell 17 ###

def bar_chart_multiple_years_26(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_26(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_26(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_26(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_26(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_26(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_26(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_26(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_26(responses_df_2018, question_of_interest)
        add_year_column_to_dataframes_26(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_26(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_26(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_26(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_26(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_26(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data_26(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_26(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)

question_of_interest = 'On which platforms have you begun or completed data science courses'
question_of_interest_alternate = 'On which online platforms have you begun or completed data science courses'
responses_df_2018.columns = responses_df_2018.columns.str.replace(question_of_interest_alternate, question_of_interest)
responses_df_2018.replace(['Kaggle Learn'], 'Kaggle Learn Courses', inplace=True)
responses_df_2018.columns = responses_df_2018.columns.str.replace('Kaggle Learn', 'Kaggle Learn Courses', regex=False)
responses_df_2018.replace(['Fast.AI'], 'Fast.ai', inplace=True)
responses_df_2018.columns = responses_df_2018.columns.str.replace('Fast.AI', 'Fast.ai')
responses_df_2018.replace(['Online University Courses'], 'University Courses (resulting in a university degree)', inplace=True)
responses_df_2018.columns = responses_df_2018.columns.str.replace('Online University Courses', 'University Courses (resulting in a university degree)', regex=False)
responses_df_2019.replace(['Kaggle Courses (i.e. Kaggle Learn)'], 'Kaggle Learn Courses', inplace=True)
responses_df_2019.columns = responses_df_2019.columns.str.replace('Kaggle Courses (i.e. Kaggle Learn)', 'Kaggle Learn Courses', regex=False)
question_of_interest = 'On which platforms have you begun or completed data science courses?'
title_for_chart = 'Most popular learning platforms (2018-2022)'
title_for_y_axis = '% of respondents'
(learning_platform_df_combined, learning_platform_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(question_of_interest)
learning_platform_df_combined_percentages = convert_df_of_counts_to_percentages_26(learning_platform_df_combined, learning_platform_df_combined_counts)
learning_platform_df_combined_percentages.columns = learning_platform_df_combined_percentages.columns.str.replace('(resulting in a university degree)', '', regex=False)
learning_platform_df_combined_percentages = learning_platform_df_combined_percentages.loc[:, ['year', 'Coursera', 'University Courses ', 'Kaggle Learn Courses', 'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other']]
df = learning_platform_df_combined_percentages.melt(id_vars=['year'], value_vars=['Coursera', 'University Courses ', 'Kaggle Learn Courses', 'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai'])
df = df.sort_values(by=['year', 'value'], ascending=True)
df.columns = df.columns.str.replace('variable', '')
bar_chart_multiple_years_26(df, '', 'value', title_for_chart, title_for_y_axis)

CPU times: user 2.72 s, sys: 328 ms, total: 3.04 s
Wall time: 2.99 s


In [23]:
%%time
### cell 18 ###

def bar_chart_multiple_choice_multiple_selection_27(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_27(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'What products or platforms did you find to be most helpful when you first started studying data science?'
title_for_chart = 'Most helpful platforms for data science education'
online_learning_platforms_df_2022 = grab_subset_of_data_27(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_27(online_learning_platforms_df_2022, title_for_chart, orientation='h')

CPU times: user 163 ms, sys: 20.9 ms, total: 184 ms
Wall time: 177 ms


In [24]:
%%time
### cell 19 ###

def bar_chart_multiple_choice_28(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_28(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'What is the highest level of formal education that you have attained or plan to attain within the next 2 years?'
responses_df_2022[question_of_interest].replace(['Bachelorâ\x80\x99s degree'], "Bachelor's degree", inplace=True)
responses_df_2022[question_of_interest].replace(['Masterâ\x80\x99s degree'], "Master's degree", inplace=True)
responses_df_2022[question_of_interest].replace(['Some college/university study without earning a bachelorâ\x80\x99s degree'], 'Some university study but without earning a degree', inplace=True)
responses_in_order = ['No formal education past high school', 'Some university study but without earning a degree', "Bachelor's degree", "Master's degree", 'Doctoral degree', 'Professional doctorate']
percentages = count_then_return_percent_28(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Most common degree types on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_28(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 170 ms, sys: 8.7 ms, total: 179 ms
Wall time: 170 ms


In [25]:
%%time
### cell 20 ###

def bar_chart_multiple_years_29(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_29(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_29(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_29(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_29(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_29(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_29(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_29(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_29(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_29(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_29(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_29(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_29(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_29(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_29(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_29(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_29(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

question_of_interest = 'What is the highest level of formal education that you have attained or plan to attain within the next 2 years?'
responses_df_2017.rename(columns={'FormalEducation': question_of_interest}, inplace=True)
responses_df_2017[question_of_interest].replace(["Master's degree"], "Master's degree", inplace=True)
responses_df_2017[question_of_interest].replace(["Bachelor's degree"], "Bachelor's degree", inplace=True)
responses_df_2018[question_of_interest].replace(['Masterâ\x80\x99s degree'], "Master's degree", inplace=True)
responses_df_2019[question_of_interest].replace(['Masterâ\x80\x99s degree'], "Master's degree", inplace=True)
responses_df_2020[question_of_interest].replace(['Masterâ\x80\x99s degree'], "Master's degree", inplace=True)
responses_df_2021[question_of_interest].replace(['Masterâ\x80\x99s degree'], "Master's degree", inplace=True)
responses_df_2018[question_of_interest].replace(['Bachelorâ\x80\x99s degree'], "Bachelor's degree", inplace=True)
responses_df_2019[question_of_interest].replace(['Bachelorâ\x80\x99s degree'], "Bachelor's degree", inplace=True)
responses_df_2020[question_of_interest].replace(['Bachelorâ\x80\x99s degree'], "Bachelor's degree", inplace=True)
responses_df_2021[question_of_interest].replace(['Bachelorâ\x80\x99s degree'], "Bachelor's degree", inplace=True)
question_of_interest = 'What is the highest level of formal education that you have attained or plan to attain within the next 2 years?'
column_of_interest = 'percentage'
title_for_chart = 'Most common degree types on Kaggle from 2017-2022'
title_for_x_axis = ''
title_for_y_axis = '% of respondents'
orientation_for_chart = 'v'
degree_df_combined = combine_subset_of_data_from_multiple_years_29(question_of_interest, title_for_x_axis, include_2017='yes')
degree_df_combined = degree_df_combined.sort_values(by=['year', 'percentage'], ascending=True)
subset_of_degrees = ["Bachelor's degree", "Master's degree", 'Doctoral degree']
degree_df_combined[''][~degree_df_combined[''].isin(subset_of_degrees)] = 'Other'
bar_chart_multiple_years_29(degree_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 632 ms, sys: 30.1 ms, total: 663 ms
Wall time: 629 ms


In [26]:
%%time
### cell 21 ###

def bar_chart_multiple_choice_30(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_30(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Have you ever published any academic research (papers, preprints, conference proceedings, etc)?'
percentages = count_then_return_percent_30(responses_df_2022, question_of_interest).sort_index()
title_for_chart = 'Proportion of advanced degree holders that have published academic research'
title_for_y_axis = '% of respondents'
orientation_for_chart = 'h'
bar_chart_multiple_choice_30(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=2)

CPU times: user 78.7 ms, sys: 228 μs, total: 79 ms
Wall time: 74.1 ms


In [27]:
%%time
### cell 22 ###

def bar_chart_multiple_choice_multiple_selection_31(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_31(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Did your research make use of machine learning?'
title_for_chart = 'Academic research topics'
online_learning_platforms_df_2022 = grab_subset_of_data_31(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_31(online_learning_platforms_df_2022, title_for_chart)

CPU times: user 143 ms, sys: 32.5 ms, total: 175 ms
Wall time: 168 ms


In [28]:
%%time
### cell 23 ###

def bar_chart_multiple_choice_32(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_32(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'For how many years have you been writing code and/or programming?'
responses_in_order = ['I have never written code', '< 1 years', '1-3 years', '3-5 years', '5-10 years', '10-20 years', '20+ years']
percentages = count_then_return_percent_32(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
orientation_for_chart = 'h'
title_for_chart = 'Programming Experience on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_32(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 78 ms, sys: 4.71 ms, total: 82.8 ms
Wall time: 76.2 ms


In [29]:
%%time
### cell 24 ###

def bar_chart_multiple_years_33(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_33(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_33(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_33(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_33(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_33(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_33(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_33(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_33(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_33(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_33(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_33(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_33(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_33(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_33(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_33(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_33(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

question_of_interest = 'For how many years have you been writing code and/or programming?'
responses_df_2018.rename(columns={'How long have you been writing code to analyze data?': question_of_interest}, inplace=True)
responses_df_2018[question_of_interest].replace(['< 1 year'], '< 1 years', inplace=True)
responses_df_2018[question_of_interest].replace(['I have never written code but I want to learn', 'I have never written code and I do not want to learn'], 'I have never written code', inplace=True)
responses_df_2018[question_of_interest].replace(['1-2 years'], '1-3 years', inplace=True)
responses_df_2018[question_of_interest].replace(['20-30 years'], '20+ years', inplace=True)
responses_df_2018[question_of_interest].replace(['30-40 years'], '20+ years', inplace=True)
responses_df_2018[question_of_interest].replace(['40+ years'], '20+ years', inplace=True)
responses_df_2019.rename(columns={'How long have you been writing code to analyze data (at work or at school)?': question_of_interest}, inplace=True)
responses_df_2019[question_of_interest].replace(['1-2 years'], '1-3 years', inplace=True)
responses_df_2020[question_of_interest].replace(['1-2 years'], '1-3 years', inplace=True)
title_for_chart = 'Programming Experience on Kaggle (2018-2022)'
title_for_y_axis = '% of respondents'
title_for_x_axis = ''
programming_df_combined = combine_subset_of_data_from_multiple_years_33(question_of_interest, title_for_x_axis).sort_values(by=['year', 'percentage'], ascending=True)
bar_chart_multiple_years_33(programming_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 604 ms, sys: 38.5 ms, total: 643 ms
Wall time: 615 ms


In [30]:
%%time
### cell 25 ###

def bar_chart_multiple_choice_multiple_selection_34(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_34(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'What programming languages do you use on a regular basis?'
title_for_chart = 'Most Popular Programming Languages'
online_learning_platforms_df_2022 = grab_subset_of_data_34(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_34(online_learning_platforms_df_2022, title_for_chart, orientation='h')

CPU times: user 163 ms, sys: 35.5 ms, total: 198 ms
Wall time: 192 ms


In [31]:
%%time
### cell 26 ###

def bar_chart_multiple_years_35(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_35(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_35(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_35(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_35(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_35(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_35(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_35(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_35(responses_df_2018, question_of_interest)
        add_year_column_to_dataframes_35(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_35(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_35(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_35(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_35(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_35(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data_35(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_35(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)

question_of_interest = 'What programming languages do you use on a regular basis?'
title_for_chart = 'Most Popular Programming Languages 2018-2022'
title_for_y_axis = '% of respondents'
(language_df_combined, language_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(question_of_interest)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(language_df_combined, language_df_combined_counts)
language_df_combined_percentages = language_df_combined_percentages.loc[:, ['year', 'Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']]
df = language_df_combined_percentages.melt(id_vars=['year'], value_vars=['Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other'])
df = df.sort_values(by=['year', 'value'], ascending=True)
df = df.rename(columns={'variable': ''})
bar_chart_multiple_years_35(df, '', 'value', title_for_chart, title_for_y_axis)

CPU times: user 1.17 s, sys: 135 ms, total: 1.31 s
Wall time: 1.28 s


In [32]:
%%time
### cell 27 ###

def bar_chart_multiple_choice_multiple_selection_36(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_36(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
ide_df_2022 = grab_subset_of_data_36(responses_df_2022, question_of_interest)
title_for_chart = "Most Popular IDE's"
bar_chart_multiple_choice_multiple_selection_36(ide_df_2022, title_for_chart, orientation='h')

CPU times: user 177 ms, sys: 20.1 ms, total: 197 ms
Wall time: 190 ms


In [33]:
%%time
### cell 28 ###

def bar_chart_multiple_choice_multiple_selection_37(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_37(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
ide_df_2022 = grab_subset_of_data_37(responses_df_2022, question_of_interest)
columns_to_combine = ['Jupyter Notebook', 'JupyterLab ']
ide_df_2022['Jupyter/JupyterLab'] = ide_df_2022[columns_to_combine].notna().any(axis='columns')
ide_df_2022 = ide_df_2022.drop(columns=columns_to_combine)
ide_df_2022['Jupyter/JupyterLab'].replace([True], 'Jupyter/JupyterLab', inplace=True)
ide_df_2022['Jupyter/JupyterLab'].replace([False], np.nan, inplace=True)
columns_to_combine = ['Visual Studio Code (VSCode) ', 'Visual Studio ']
ide_df_2022['Visual Studio / Visual Studio Code (VSCode)'] = ide_df_2022[columns_to_combine].notna().any(axis='columns')
ide_df_2022 = ide_df_2022.drop(columns=columns_to_combine)
ide_df_2022['Visual Studio / Visual Studio Code (VSCode)'].replace([True], 'Visual Studio / Visual Studio Code (VSCode)', inplace=True)
ide_df_2022['Visual Studio / Visual Studio Code (VSCode)'].replace([False], np.nan, inplace=True)
title_for_chart = "Most Popular IDE's (after grouping similar products)"
bar_chart_multiple_choice_multiple_selection_37(ide_df_2022, title_for_chart, orientation='h')

CPU times: user 366 ms, sys: 20.4 ms, total: 386 ms
Wall time: 379 ms


In [34]:
%%time
### cell 29 ###

def bar_chart_multiple_years_38(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_38(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_38(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_38(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_38(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_38(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_38(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_38(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_38(responses_df_2018, question_of_interest)
        add_year_column_to_dataframes_38(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_38(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_38(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_38(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_38(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_38(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data_38(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_38(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)

correct_phrasing = 'Jupyter Notebook / JupyterLab'
incorrect_phrasing = 'Jupyter/IPython'
responses_df_2018.columns = responses_df_2018.columns.str.replace(incorrect_phrasing, correct_phrasing, regex=False)
question_of_interest = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
alternate_phrasing = "Which of the following integrated development environments (IDE's) have you used at work or school in the last 5 years?"
responses_df_2018.columns = responses_df_2018.columns.str.replace(alternate_phrasing, question_of_interest, regex=False)
title_for_chart = "Most Popular IDE's 2018-2022"
x_axis_title = 'Percentage of respondents'
(ide_df_combined, ide_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(question_of_interest)
ide_df_combined_2 = ide_df_combined.copy()
columns_to_combine = ['Jupyter Notebook', 'JupyterLab ', 'Jupyter (JupyterLab, Jupyter Notebooks, etc) ', 'Jupyter Notebook / JupyterLab']
ide_df_combined_2['Jupyter Notebook / JupyterLab_'] = ide_df_combined_2[columns_to_combine].notna().any(axis='columns')
ide_df_combined_2 = ide_df_combined_2.drop(columns=columns_to_combine)
ide_df_combined_2['Jupyter Notebook / JupyterLab_'].replace([True], 'Jupyter Notebook / JupyterLab', inplace=True)
ide_df_combined_2['Jupyter Notebook / JupyterLab_'].replace([False], np.nan, inplace=True)
ide_df_combined_2.columns = ide_df_combined_2.columns.str.replace('Jupyter Notebook / JupyterLab_', 'Jupyter Notebook / JupyterLab', regex=False)
columns_to_combine = ['MATLAB', 'MATLAB ']
ide_df_combined_2['MATLAB_'] = ide_df_combined_2[columns_to_combine].notna().any(axis='columns')
ide_df_combined_2 = ide_df_combined_2.drop(columns=columns_to_combine)
ide_df_combined_2['MATLAB_'].replace([True], 'MATLAB', inplace=True)
ide_df_combined_2['MATLAB_'].replace([False], np.nan, inplace=True)
ide_df_combined_2.columns = ide_df_combined_2.columns.str.replace('MATLAB_', 'MATLAB', regex=False)
columns_to_combine = ['RStudio', 'RStudio ']
ide_df_combined_2['RStudio_'] = ide_df_combined_2[columns_to_combine].notna().any(axis='columns')
ide_df_combined_2 = ide_df_combined_2.drop(columns=columns_to_combine)
ide_df_combined_2['RStudio_'].replace([True], 'RStudio', inplace=True)
ide_df_combined_2['RStudio_'].replace([False], np.nan, inplace=True)
ide_df_combined_2.columns = ide_df_combined_2.columns.str.replace('RStudio_', 'RStudio', regex=False)
columns_to_combine = ['Visual Studio Code', 'Visual Studio', 'Visual Studio / Visual Studio Code ', 'Visual Studio ', 'Visual Studio Code (VSCode) ', 'Click to write Choice 13']
ide_df_combined_2['Visual Studio / Visual Studio Code (VSCode)'] = ide_df_combined_2[columns_to_combine].notna().any(axis='columns')
ide_df_combined_2 = ide_df_combined_2.drop(columns=columns_to_combine)
ide_df_combined_2['Visual Studio / Visual Studio Code (VSCode)'].replace([True], 'Visual Studio / Visual Studio Code (VSCode)', inplace=True)
ide_df_combined_2['Visual Studio / Visual Studio Code (VSCode)'].replace([False], np.nan, inplace=True)
columns_to_combine = ['PyCharm ', 'PyCharm']
ide_df_combined_2['PyCharm_'] = ide_df_combined_2[columns_to_combine].notna().any(axis='columns')
ide_df_combined_2 = ide_df_combined_2.drop(columns=columns_to_combine)
ide_df_combined_2['PyCharm_'].replace([True], 'PyCharm', inplace=True)
ide_df_combined_2['PyCharm_'].replace([False], np.nan, inplace=True)
ide_df_combined_2.columns = ide_df_combined_2.columns.str.replace('PyCharm_', 'PyCharm', regex=False)
ide_df_combined_counts_2 = ide_df_combined_2.groupby('year').count().reset_index()
ide_df_combined_percentages = convert_df_of_counts_to_percentages_38(ide_df_combined_2, ide_df_combined_counts_2)
ide_df_combined_percentages = ide_df_combined_percentages.loc[:, ['year', 'Jupyter Notebook / JupyterLab', 'Visual Studio / Visual Studio Code (VSCode)', 'RStudio', 'PyCharm', 'MATLAB']]
df = ide_df_combined_percentages.melt(id_vars=['year'], value_vars=['Jupyter Notebook / JupyterLab', 'Visual Studio / Visual Studio Code (VSCode)', 'RStudio', 'PyCharm', 'MATLAB'])
df = df.sort_values(by=['year', 'value'], ascending=True)
df = df.rename(columns={'variable': ''})
bar_chart_multiple_years_38(df, '', 'value', title_for_chart, title_for_y_axis)

CPU times: user 1.84 s, sys: 162 ms, total: 2.01 s
Wall time: 1.96 s


In [35]:
%%time
### cell 30 ###

def bar_chart_multiple_choice_multiple_selection_39(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_39(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following hosted notebook products?'
title_for_chart = 'Most popular hosted notebooks in 2022'
notebooks_df_2022 = grab_subset_of_data_39(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_39(notebooks_df_2022, title_for_chart, orientation='h')

CPU times: user 189 ms, sys: 8.54 ms, total: 198 ms
Wall time: 190 ms


In [36]:
%%time
### cell 31 ###

def bar_chart_multiple_years_40(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_40(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_40(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018, question_of_interest)
        add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data_40(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)

question_of_interest_original = 'Which of the following hosted notebook products do you use on a regular basis?'
question_of_interest_new = 'Do you use any of the following hosted notebook products?'
responses_df_2021.columns = responses_df_2021.columns.str.replace(question_of_interest_original, question_of_interest_new, regex=False)
notebooks_df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest_new)
question_of_interest_original = 'Which of the following hosted notebook products do you use on a regular basis?'
question_of_interest_new = 'Do you use any of the following hosted notebook products?'
responses_df_2020.columns = responses_df_2020.columns.str.replace(question_of_interest_original, question_of_interest_new, regex=False)
notebooks_df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest_new)
colab_text_to_replace = 'Google Colab '
colab_new_text = 'Colab Notebooks'
colab_answer = 'Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Google Colab '
kaggle_text_to_replace = 'Kaggle Notebooks (Kernels) '
kaggle_new_text = 'Kaggle Notebooks'
kaggle_answer = 'Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Kaggle Notebooks (Kernels) '
responses_df_2019[colab_answer] = responses_df_2019[colab_answer].str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2019.columns = responses_df_2019.columns.str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2019[kaggle_answer] = responses_df_2019[kaggle_answer].str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
responses_df_2019.columns = responses_df_2019.columns.str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
question_of_interest_original = 'Which of the following hosted notebook products do you use on a regular basis?'
question_of_interest_new = 'Do you use any of the following hosted notebook products?'
responses_df_2019.columns = responses_df_2019.columns.str.replace(question_of_interest_original, question_of_interest_new, regex=False)
notebooks_df_2019 = grab_subset_of_data_40(responses_df_2019, question_of_interest_new)
colab_text_to_replace = 'Google Colab'
colab_new_text = 'Colab Notebooks'
colab_answer = 'Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Google Colab'
kaggle_text_to_replace = 'Kaggle Kernels'
kaggle_new_text = 'Kaggle Notebooks'
kaggle_answer = 'Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Kaggle Kernels'
responses_df_2018[colab_answer] = responses_df_2018[colab_answer].str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2018.columns = responses_df_2018.columns.str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2018[kaggle_answer] = responses_df_2018[kaggle_answer].str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
responses_df_2018.columns = responses_df_2018.columns.str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
question_of_interest_original = 'Which of the following hosted notebooks have you used at work or school in the last 5 years?'
question_of_interest_new = 'Do you use any of the following hosted notebook products?'
responses_df_2018.columns = responses_df_2018.columns.str.replace(question_of_interest_original, question_of_interest_new, regex=False)
notebooks_df_2018 = grab_subset_of_data_40(responses_df_2018, question_of_interest_new)
question_of_interest = 'Do you use any of the following hosted notebook products?'
(notebooks_df_combined, notebooks_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(question_of_interest)
notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(notebooks_df_combined, notebooks_df_combined_counts)
notebooks_df_combined_percentages = notebooks_df_combined_percentages.loc[:, ['year', 'None', 'Kaggle Notebooks', 'Colab Notebooks']]
title_for_chart = 'Most popular hosted notebooks products 2018-2022'
title_for_y_axis = '% of respondents'
df = notebooks_df_combined_percentages.melt(id_vars=['year'], value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks'])
df = df.rename(columns={'variable': ''})
bar_chart_multiple_years_40(df, '', 'value', title_for_chart, title_for_y_axis)

CPU times: user 2.64 s, sys: 574 ms, total: 3.21 s
Wall time: 3.14 s


In [37]:
%%time
### cell 32 ###

def bar_chart_multiple_choice_multiple_selection_41(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_41(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following data visualization libraries on a regular basis?'
title_for_chart = 'Most popular data visualization frameworks'
notebooks_df_2022 = grab_subset_of_data_41(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_41(notebooks_df_2022, title_for_chart, orientation='h')

CPU times: user 176 ms, sys: 20.4 ms, total: 197 ms
Wall time: 189 ms


In [38]:
%%time
### cell 33 ###

def bar_chart_multiple_choice_42(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_42(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'For how many years have you used machine learning methods?'
responses_in_order = ['I do not use machine learning methods', 'Under 1 year', '1-2 years', '2-3 years', '3-4 years', '4-5 years', '5-10 years', '10-20 years']
percentages = count_then_return_percent_42(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'ML Experience Levels on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_42(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 81.2 ms, sys: 545 μs, total: 81.7 ms
Wall time: 75 ms


In [39]:
%%time
### cell 34 ###

def bar_chart_multiple_years_43(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_43(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_43(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_43(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_43(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_43(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_43(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_43(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_43(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_43(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_43(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_43(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_43(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_43(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_43(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_43(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_43(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

question_of_interest = 'For how many years have you used machine learning methods?'
responses_df_2018.rename(columns={'For how many years have you used machine learning methods (at work or in school)?': question_of_interest}, inplace=True)
responses_df_2018[question_of_interest].replace(['< 1 year'], 'Under 1 year', inplace=True)
responses_df_2018[question_of_interest].replace(['10-15 years'], '10-20 years', inplace=True)
responses_df_2018[question_of_interest].replace(['20+ years'], '10-20 years', inplace=True)
responses_df_2018[question_of_interest].replace(['I have never studied machine learning but plan to learn in the future'], 'I do not use machine learning methods', inplace=True)
responses_df_2018[question_of_interest].replace(['I have never studied machine learning and I do not plan to'], 'I do not use machine learning methods', inplace=True)
responses_df_2019[question_of_interest].replace(['< 1 years'], 'Under 1 year', inplace=True)
responses_df_2019[question_of_interest].replace(['10-15 years'], '10-20 years', inplace=True)
responses_df_2019[question_of_interest].replace(['20+ years'], '20 or more years', inplace=True)
title_for_chart = 'ML experience for Kagglers (2018-2022)'
title_for_y_axis = '% of respondents'
title_for_x_axis = ''
ml_exp_df_combined = combine_subset_of_data_from_multiple_years_43(question_of_interest, title_for_x_axis).sort_values(by=['year', 'percentage'], ascending=True)
bar_chart_multiple_years_43(ml_exp_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 549 ms, sys: 26.1 ms, total: 575 ms
Wall time: 547 ms


In [40]:
%%time
### cell 35 ###

def bar_chart_multiple_choice_multiple_selection_44(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_44(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Which of the following machine learning frameworks do you use on a regular basis?'
title_for_chart = 'Most popular machine learning frameworks'
ml_frameworks_df_2022 = grab_subset_of_data_44(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_44(ml_frameworks_df_2022, title_for_chart, orientation='h')

CPU times: user 169 ms, sys: 24.7 ms, total: 194 ms
Wall time: 187 ms


In [41]:
%%time
### cell 36 ###

def bar_chart_multiple_choice_multiple_selection_45(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_45(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

ml_frameworks_df_2022 = grab_subset_of_data_45(responses_df_2022, question_of_interest)
columns_to_combine = ['TensorFlow ', 'Keras ']
ml_frameworks_df_2022['TensorFlow/Keras'] = ml_frameworks_df_2022[columns_to_combine].notna().any(axis='columns')
ml_frameworks_df_2022 = ml_frameworks_df_2022.drop(columns=columns_to_combine)
ml_frameworks_df_2022['TensorFlow/Keras'].replace([True], 'TensorFlow/Keras', inplace=True)
ml_frameworks_df_2022['TensorFlow/Keras'].replace([False], np.nan, inplace=True)
columns_to_combine = ['PyTorch ', 'PyTorch Lightning ', 'Fast.ai ']
ml_frameworks_df_2022['PyTorch/PyTorch Lightning/Fast.ai'] = ml_frameworks_df_2022[columns_to_combine].notna().any(axis='columns')
ml_frameworks_df_2022 = ml_frameworks_df_2022.drop(columns=columns_to_combine)
ml_frameworks_df_2022['PyTorch/PyTorch Lightning/Fast.ai'].replace([True], 'PyTorch/PyTorch Lightning/Fast.ai', inplace=True)
ml_frameworks_df_2022['PyTorch/PyTorch Lightning/Fast.ai'].replace([False], np.nan, inplace=True)
columns_to_combine = ['Xgboost ', 'LightGBM ', 'CatBoost ']
ml_frameworks_df_2022['Xgboost/LightGBM/CatBoost'] = ml_frameworks_df_2022[columns_to_combine].notna().any(axis='columns')
ml_frameworks_df_2022 = ml_frameworks_df_2022.drop(columns=columns_to_combine)
ml_frameworks_df_2022['Xgboost/LightGBM/CatBoost'].replace([True], 'Xgboost/LightGBM/CatBoost', inplace=True)
ml_frameworks_df_2022['Xgboost/LightGBM/CatBoost'].replace([False], np.nan, inplace=True)
ml_frameworks_df_2022 = ml_frameworks_df_2022.loc[:, ['Scikit—learn ', 'TensorFlow/Keras', 'PyTorch/PyTorch Lightning/Fast.ai', 'Xgboost/LightGBM/CatBoost']]
title_for_chart = 'Most popular consolidated machine learning frameworks'
bar_chart_multiple_choice_multiple_selection_45(ml_frameworks_df_2022, title_for_chart, orientation='h')

CPU times: user 243 ms, sys: 23.3 ms, total: 267 ms
Wall time: 258 ms


In [42]:
%%time
### cell 37 ###

def bar_chart_multiple_years_46(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_46(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_46(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_46(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_46(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_46(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_46(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_46(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_46(responses_df_2018, question_of_interest)
        add_year_column_to_dataframes_46(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_46(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_46(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_46(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_46(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_46(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data_46(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_46(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)

question_of_interest_old = 'What machine learning frameworks have you used in the past 5 years?'
question_of_interest_new = 'Which of the following machine learning frameworks do you use on a regular basis?'
responses_df_2018.columns = responses_df_2018.columns.str.replace(question_of_interest_old, question_of_interest_new, regex=False)
question_of_interest = 'Which of the following machine learning frameworks do you use on a regular basis?'
(ml_df_combined, ml_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(question_of_interest)
columns_to_combine = ['TensorFlow', 'TensorFlow ']
ml_df_combined['TensorFlow_'] = ml_df_combined[columns_to_combine].notna().any(axis='columns')
ml_df_combined = ml_df_combined.drop(columns=columns_to_combine)
ml_df_combined['TensorFlow_'].replace([True], 'TensorFlow', inplace=True)
ml_df_combined['TensorFlow_'].replace([False], np.nan, inplace=True)
ml_df_combined.columns = ml_df_combined.columns.str.replace('TensorFlow_', 'TensorFlow ', regex=False)
columns_to_combine = ['Keras', 'Keras ']
ml_df_combined['Keras_'] = ml_df_combined[columns_to_combine].notna().any(axis='columns')
ml_df_combined = ml_df_combined.drop(columns=columns_to_combine)
ml_df_combined['Keras_'].replace([True], 'Keras', inplace=True)
ml_df_combined['Keras_'].replace([False], np.nan, inplace=True)
ml_df_combined.columns = ml_df_combined.columns.str.replace('Keras_', 'Keras ', regex=False)
columns_to_combine = ['PyTorch', 'PyTorch ']
ml_df_combined['PyTorch_'] = ml_df_combined[columns_to_combine].notna().any(axis='columns')
ml_df_combined = ml_df_combined.drop(columns=columns_to_combine)
ml_df_combined['PyTorch_'].replace([True], 'PyTorch', inplace=True)
ml_df_combined['PyTorch_'].replace([False], np.nan, inplace=True)
ml_df_combined.columns = ml_df_combined.columns.str.replace('PyTorch_', 'PyTorch ', regex=False)
columns_to_combine = ['Scikit—learn ', 'Learn', 'learn ']
ml_df_combined['Scikit—learn_'] = ml_df_combined[columns_to_combine].notna().any(axis='columns')
ml_df_combined = ml_df_combined.drop(columns=columns_to_combine)
ml_df_combined['Scikit—learn_'].replace([True], 'Scikit—learn', inplace=True)
ml_df_combined['Scikit—learn_'].replace([False], np.nan, inplace=True)
ml_df_combined.columns = ml_df_combined.columns.str.replace('Scikit—learn_', 'Scikit-learn ', regex=False)
ml_df_combined_counts_2 = ml_df_combined.groupby('year').count().reset_index()
ml_df_combined_percentages = convert_df_of_counts_to_percentages_46(ml_df_combined, ml_df_combined_counts_2)
ml_df_combined_percentages = ml_df_combined_percentages.loc[:, ['year', 'Scikit-learn ', 'TensorFlow ', 'Keras ', 'PyTorch ', 'None', 'Other']]
title_for_chart = 'Most Popular Machine Learning Frameworks 2018-2022'
title_for_y_axis = '% of respondents'
df = ml_df_combined_percentages.melt(id_vars=['year'], value_vars=['Scikit-learn ', 'TensorFlow ', 'Keras ', 'PyTorch ', 'None', 'Other'])
df = df.sort_values(by=['year', 'value'], ascending=True)
df = df.rename(columns={'variable': ''})
bar_chart_multiple_years_46(df, '', 'value', title_for_chart, title_for_y_axis)

CPU times: user 1.73 s, sys: 308 ms, total: 2.03 s
Wall time: 2 s


In [43]:
%%time
### cell 38 ###

def bar_chart_multiple_years_47(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_47(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_47(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_47(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_47(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_47(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_47(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_47(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_47(responses_df_2018, question_of_interest)
        add_year_column_to_dataframes_47(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_47(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_47(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_47(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_47(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_47(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data_47(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_47(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)

question_of_interest_old = 'What machine learning frameworks have you used in the past 5 years?'
question_of_interest_new = 'Which of the following machine learning frameworks do you use on a regular basis?'
responses_df_2018.columns = responses_df_2018.columns.str.replace(question_of_interest_old, question_of_interest_new, regex=False)
question_of_interest = 'Which of the following machine learning frameworks do you use on a regular basis?'
(ml_df_combined, ml_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(question_of_interest)
ml_df_combined_2 = ml_df_combined.copy()
columns_to_combine = ['TensorFlow', 'TensorFlow ', 'Keras', 'Keras ']
ml_df_combined_2['TensorFlow/Keras'] = ml_df_combined_2[columns_to_combine].notna().any(axis='columns')
ml_df_combined_2 = ml_df_combined_2.drop(columns=columns_to_combine)
ml_df_combined_2['TensorFlow/Keras'].replace([True], 'TensorFlow/Keras', inplace=True)
ml_df_combined_2['TensorFlow/Keras'].replace([False], np.nan, inplace=True)
columns_to_combine = ['PyTorch', 'PyTorch ', 'PyTorch Lightning ', 'Fast.ai ', 'Fastai']
ml_df_combined_2['PyTorch/Lightning/Fast.ai'] = ml_df_combined_2[columns_to_combine].notna().any(axis='columns')
ml_df_combined_2 = ml_df_combined_2.drop(columns=columns_to_combine)
ml_df_combined_2['PyTorch/Lightning/Fast.ai'].replace([True], 'PyTorch/PyTorch Lightning/Fast.ai', inplace=True)
ml_df_combined_2['PyTorch/Lightning/Fast.ai'].replace([False], np.nan, inplace=True)
columns_to_combine = ['Xgboost', 'Xgboost ', 'lightgbm', 'LightGBM ', 'catboost', 'CatBoost ']
ml_df_combined_2['Xgboost/LightGBM/CatBoost'] = ml_df_combined_2[columns_to_combine].notna().any(axis='columns')
ml_df_combined_2 = ml_df_combined_2.drop(columns=columns_to_combine)
ml_df_combined_2['Xgboost/LightGBM/CatBoost'].replace([True], 'Xgboost/LightGBM/CatBoost', inplace=True)
ml_df_combined_2['Xgboost/LightGBM/CatBoost'].replace([False], np.nan, inplace=True)
columns_to_combine = ['Scikit—learn ', 'Scikit—learn ', 'learn ', 'Learn']
ml_df_combined_2['Scikit-learn_'] = ml_df_combined_2[columns_to_combine].notna().any(axis='columns')
ml_df_combined_2 = ml_df_combined_2.drop(columns=columns_to_combine)
ml_df_combined_2['Scikit-learn_'].replace([True], 'Scikit-learn_', inplace=True)
ml_df_combined_2['Scikit-learn_'].replace([False], np.nan, inplace=True)
ml_df_combined_2.columns = ml_df_combined_2.columns.str.replace('Scikit-learn_', 'Scikit-learn', regex=False)
ml_df_combined_counts_2 = ml_df_combined_2.groupby('year').count().reset_index()
ml_df_combined_percentages = convert_df_of_counts_to_percentages_47(ml_df_combined_2, ml_df_combined_counts_2)
ml_df_combined_percentages = ml_df_combined_percentages.loc[:, ['year', 'Scikit-learn', 'TensorFlow/Keras', 'PyTorch/Lightning/Fast.ai', 'Xgboost/LightGBM/CatBoost']]
df = ml_df_combined_percentages.melt(id_vars=['year'], value_vars=['Scikit-learn', 'Xgboost/LightGBM/CatBoost', 'TensorFlow/Keras', 'PyTorch/Lightning/Fast.ai'])
df = df.sort_values(by=['year', 'value'], ascending=True)
df = df.rename(columns={'variable': ''})
title_for_chart = 'Most popular consolidated machine learning frameworks (2018-2022)'
bar_chart_multiple_years_47(df, '', 'value', title_for_chart, title_for_y_axis)

CPU times: user 1.7 s, sys: 390 ms, total: 2.09 s
Wall time: 2.06 s


In [44]:
%%time
### cell 39 ###

def bar_chart_multiple_choice_multiple_selection_48(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_48(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Which of the following ML algorithms do you use on a regular basis?'
title_for_chart = 'Most popular machine learning algorithms'
ml_algos_df_2022 = grab_subset_of_data_48(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_48(ml_algos_df_2022, title_for_chart, orientation='h')

CPU times: user 178 ms, sys: 20.4 ms, total: 199 ms
Wall time: 192 ms


In [45]:
%%time
### cell 40 ###

def bar_chart_multiple_choice_multiple_selection_49(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_49(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Which categories of computer vision methods do you use on a regular basis?'
title_for_chart = 'Most popular computer vision methods'
cv_df_2022 = grab_subset_of_data_49(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_49(cv_df_2022, title_for_chart)

CPU times: user 149 ms, sys: 27.7 ms, total: 177 ms
Wall time: 170 ms


In [46]:
%%time
### cell 41 ###

def bar_chart_multiple_choice_multiple_selection_50(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_50(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Which of the following natural language processing (NLP) methods do you use on a regular basis?'
title_for_chart = 'Most popular NLP methods'
nlp_df_2022 = grab_subset_of_data_50(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_50(nlp_df_2022, title_for_chart)

CPU times: user 158 ms, sys: 24.3 ms, total: 182 ms
Wall time: 174 ms


In [47]:
%%time
### cell 42 ###

def grab_subset_of_data_51(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Which of the following natural language processing (NLP) methods do you use on a regular basis?'
nlp_df_2022 = grab_subset_of_data_51(responses_df_2022, question_of_interest)
counts_2022 = nlp_df_2022[nlp_df_2022.columns[:]].count().sort_values(ascending=True)
percentages_2022 = counts_2022 * 100 / len(nlp_df_2022)
transformers = 'Transformer language models (GPT—3, BERT, XLnet, etc)'
percentages_2022 = percentages_2022.rename({transformers: 'Transformers'})[['Transformers']]
nlp_df_2021 = grab_subset_of_data_51(responses_df_2021, question_of_interest)
counts_2021 = nlp_df_2021[nlp_df_2021.columns[:]].count().sort_values(ascending=True)
percentages_2021 = counts_2021 * 100 / len(nlp_df_2021)
transformers = '3, BERT, XLnet, etc)'
percentages_2021 = percentages_2021.rename({transformers: 'Transformers'})[['Transformers']]
nlp_df_2020 = grab_subset_of_data_51(responses_df_2020, question_of_interest)
counts_2020 = nlp_df_2020[nlp_df_2020.columns[:]].count().sort_values(ascending=True)
percentages_2020 = counts_2020 * 100 / len(nlp_df_2020)
transformers = '3, BERT, XLnet, etc)'
percentages_2020 = percentages_2020.rename({transformers: 'Transformers'})[['Transformers']]
nlp_df_2019 = grab_subset_of_data_51(responses_df_2019, question_of_interest)
counts_2019 = nlp_df_2019[nlp_df_2019.columns[:]].count().sort_values(ascending=True)
percentages_2019 = counts_2019 * 100 / len(nlp_df_2019)
transformers = '2, BERT, XLnet, etc)'
percentages_2019 = percentages_2019.rename({transformers: 'Transformers'})[['Transformers']]
title_for_chart = 'Popularity of transformer architectures for NLP tasks 2019-2022'
title_for_y_axis = '% of respondents'
percentages_2019.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2019_' + title_for_chart + '.csv', index=True)
percentages_2020.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2020_' + title_for_chart + '.csv', index=True)
percentages_2021.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2021_' + title_for_chart + '.csv', index=True)
percentages_2022.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2022_' + title_for_chart + '.csv', index=True)

CPU times: user 695 ms, sys: 119 ms, total: 814 ms
Wall time: 784 ms


In [48]:
%%time
### cell 43 ###

def bar_chart_multiple_choice_multiple_selection_52(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_52(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you download pre—trained model weights from any of the following services?'
title_for_chart = 'Most popular ML model repositories'
models_df_2022 = grab_subset_of_data_52(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_52(models_df_2022, title_for_chart, orientation='h')

CPU times: user 158 ms, sys: 26.7 ms, total: 185 ms
Wall time: 178 ms


In [49]:
%%time
### cell 44 ###

def bar_chart_multiple_choice_53(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_53(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Which of the following ML model hubs/repositories do you use most often? - Selected Choice'
responses_in_order = [' Kaggle datasets ', ' Huggingface Models ', '  TensorFlow Hub ', ' PyTorch Hub ', ' Timm ', ' NVIDIA NGC models  ', ' ONNX models ', ' Jumpstart ']
percentages = count_then_return_percent_53(responses_df_2022, question_of_interest).sort_index()[responses_in_order].iloc[::-1]
title_for_chart = 'Favorite ML model repository (for users of multiple repositories)'
title_for_y_axis = '% of respondents'
orientation_for_chart = 'h'
bar_chart_multiple_choice_53(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=9)

CPU times: user 73.9 ms, sys: 8.25 ms, total: 82.2 ms
Wall time: 75.8 ms


In [50]:
%%time
### cell 45 ###

def bar_chart_multiple_choice_54(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_54(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
percentages = count_then_return_percent_54(responses_df_2022, question_of_interest).sort_index()
percentages = percentages.sort_values(ascending=True)
title_for_chart = 'Most common job titles on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_54(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=15)

CPU times: user 73.8 ms, sys: 3.64 ms, total: 77.4 ms
Wall time: 72.8 ms


In [51]:
%%time
### cell 46 ###

def bar_chart_multiple_years_55(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_55(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_55(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_55(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_55(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_55(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_55(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_55(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_55(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_55(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_55(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_55(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_55(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_55(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_55(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_55(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_55(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

question_of_interest = 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
title_for_chart = 'Most common job titles on Kaggle (2018-2022)'
title_for_y_axis = '% of respondents'
job_df_combined = combine_subset_of_data_from_multiple_years_55(question_of_interest, title_for_x_axis)
job_df_combined.replace(['Data Analyst (Business, Marketing, Financial, Quantitative, etc)'], 'Data Analyst', inplace=True)
job_df_combined = job_df_combined.rename({'Data Analyst (Business, Marketing, Financial, Quantitative, etc)': 'Data Analyst'})
job_df_combined = job_df_combined[job_df_combined.index.isin(['Data Analyst', 'Data Engineer', 'Data Scientist', 'Research Scientist', 'Software Engineer'])]
bar_chart_multiple_years_55(job_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 354 ms, sys: 44.9 ms, total: 398 ms
Wall time: 378 ms


In [52]:
%%time
### cell 47 ###

def bar_chart_multiple_choice_56(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_56(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'In what industry is your current employer/contract (or your most recent employer if retired)? - Selected Choice'
percentages = count_then_return_percent_56(responses_df_2022, question_of_interest).sort_index()
percentages = percentages.sort_values(ascending=True)
title_for_chart = 'Most common industries of employment on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_56(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 79.5 ms, sys: 0 ns, total: 79.5 ms
Wall time: 74.3 ms


In [53]:
%%time
### cell 48 ###

def bar_chart_multiple_choice_57(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_57(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'What is the size of the company where you are employed?'
responses_in_order = ['0-49 employees', '50-249 employees', '250-999 employees', '1000-9,999 employees', '10,000 or more employees']
percentages = count_then_return_percent_57(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Most common company sizes for Kagglers'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_57(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 76.5 ms, sys: 4.54 ms, total: 81 ms
Wall time: 74.5 ms


In [54]:
%%time
### cell 49 ###

def bar_chart_multiple_choice_58(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_58(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Approximately how many individuals are responsible for data science workloads at your place of business?'
responses_in_order = ['0', '1-2', '3-4', '5-9', '10-14', '15-19', '20+']
percentages = count_then_return_percent_58(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Most common data science team sizes'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_58(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 71 ms, sys: 8.23 ms, total: 79.2 ms
Wall time: 71.8 ms


In [55]:
%%time
### cell 50 ###

def bar_chart_multiple_choice_59(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_59(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Does your current employer incorporate machine learning methods into their business?'
responses_in_order = ['No (we do not use ML methods)', 'I do not know', 'We are exploring ML methods (and may one day put a model into production)', 'We recently started using ML methods (i.e., models in production for less than 2 years)', 'We have well established ML methods (i.e., models in production for more than 2 years)']
percentages = count_then_return_percent_59(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Most common "ML maturity level" of organizations'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_59(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 72.9 ms, sys: 7.59 ms, total: 80.5 ms
Wall time: 74.8 ms


In [56]:
%%time
### cell 51 ###

def bar_chart_multiple_years_60(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_60(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

def add_year_column_to_dataframes_60(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def combine_subset_of_data_from_multiple_years_60(question_of_interest, x_axis_title, include_2017=None):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_60(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_60(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_60(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_60(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_60(responses_df_2018, question_of_interest).sort_index())
        df_2017 = pd.DataFrame(count_then_return_percent_60(responses_df_2017, question_of_interest).sort_index())
        add_year_column_to_dataframes_60(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_2017.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(count_then_return_percent_60(responses_df_2022, question_of_interest).sort_index())
        df_2021 = pd.DataFrame(count_then_return_percent_60(responses_df_2021, question_of_interest).sort_index())
        df_2020 = pd.DataFrame(count_then_return_percent_60(responses_df_2020, question_of_interest).sort_index())
        df_2019 = pd.DataFrame(count_then_return_percent_60(responses_df_2019, question_of_interest).sort_index())
        df_2018 = pd.DataFrame(count_then_return_percent_60(responses_df_2018, question_of_interest).sort_index())
        add_year_column_to_dataframes_60(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ['percentage', 'year']
        df_2021.columns = ['percentage', 'year']
        df_2020.columns = ['percentage', 'year']
        df_2019.columns = ['percentage', 'year']
        df_2018.columns = ['percentage', 'year']
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ['percentage', 'year', x_axis_title]
        return df_combined

question_of_interest = 'Does your current employer incorporate machine learning methods into their business?'
title_for_chart = 'ML maturity levels for organizations (2018-2022)'
title_for_y_axis = '% of respondents'
title_for_x_axis = ''
maturity_df_combined = combine_subset_of_data_from_multiple_years_60(question_of_interest, title_for_x_axis).sort_values(by=['year', 'percentage'], ascending=True)
bar_chart_multiple_years_60(maturity_df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 341 ms, sys: 12.8 ms, total: 353 ms
Wall time: 335 ms


In [57]:
%%time
### cell 52 ###

def bar_chart_multiple_choice_multiple_selection_61(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_61(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Select any activities that make up an important part of your role at work'
title_for_chart = 'Most common data responsibilities'
models_df_2022 = grab_subset_of_data_61(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_61(models_df_2022, title_for_chart, orientation='h')

CPU times: user 153 ms, sys: 23.6 ms, total: 176 ms
Wall time: 169 ms


In [58]:
%%time
### cell 53 ###

def bar_chart_multiple_choice_62(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_62(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'What is your current yearly compensation (approximate $USD)?'
responses_in_order = ['$0-999', '1,000-1,999', '2,000-2,999', '3,000-3,999', '4,000-4,999', '5,000-7,499', '7,500-9,999', '10,000-14,999', '15,000-19,999', '20,000-24,999', '25,000-29,999', '30,000-39,999', '40,000-49,999', '50,000-59,999', '60,000-69,999', '70,000-79,999', '80,000-89,999', '90,000-99,999', '100,000-124,999', '125,000-149,999', '150,000-199,999', '200,000-249,999', '250,000-299,999', '300,000-499,999', '$500,000-999,999', '>$1,000,000']
percentages = count_then_return_percent_62(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Global salary distributions'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_62(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 80.7 ms, sys: 443 μs, total: 81.1 ms
Wall time: 74.8 ms


In [59]:
%%time
### cell 54 ###

def bar_chart_multiple_choice_63(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_63(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'What is your current yearly compensation (approximate $USD)?'
USA_responses_df_2022 = responses_df_2022[responses_df_2022['In which country do you currently reside?'] == 'United States of America']
responses_in_order = ['$0-999', '1,000-1,999', '2,000-2,999', '3,000-3,999', '5,000-7,499', '7,500-9,999', '10,000-14,999', '15,000-19,999', '20,000-24,999', '25,000-29,999', '30,000-39,999', '40,000-49,999', '50,000-59,999', '60,000-69,999', '70,000-79,999', '80,000-89,999', '90,000-99,999', '100,000-124,999', '125,000-149,999', '150,000-199,999', '200,000-249,999', '250,000-299,999', '300,000-499,999', '$500,000-999,999', '>$1,000,000']
percentages = count_then_return_percent_63(USA_responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Salary distributions in the USA'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_63(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=20)

CPU times: user 116 ms, sys: 377 μs, total: 117 ms
Wall time: 107 ms


In [60]:
%%time
### cell 55 ###

def bar_chart_multiple_choice_64(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_64(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'What is your current yearly compensation (approximate $USD)?'
India_responses_df_2022 = responses_df_2022[responses_df_2022['In which country do you currently reside?'] == 'India']
responses_in_order = ['$0-999', '1,000-1,999', '2,000-2,999', '3,000-3,999', '4,000-4,999', '5,000-7,499', '7,500-9,999', '10,000-14,999', '15,000-19,999', '20,000-24,999', '25,000-29,999', '30,000-39,999', '40,000-49,999', '50,000-59,999', '60,000-69,999', '70,000-79,999', '80,000-89,999', '90,000-99,999', '100,000-124,999', '125,000-149,999', '150,000-199,999', '200,000-249,999', '250,000-299,999', '300,000-499,999', '$500,000-999,999', '>$1,000,000']
percentages = count_then_return_percent_64(India_responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Salary distributions in India'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_64(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=20)

CPU times: user 141 ms, sys: 3.39 ms, total: 144 ms
Wall time: 134 ms


In [61]:
%%time
### cell 56 ###

def bar_chart_multiple_choice_65(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_65(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

responses_in_order = ['$0 ($USD)', '$1-$99', '$100-$999', '$1000-$9,999', '$10,000-$99,999', '$100,000 or more ($USD)']
question_of_interest = 'Approximately how much money have you spent on machine learning and/or cloud computing services at home or at work in the past 5 years (approximate $USD)?\n (approximate $USD)?'
percentages = count_then_return_percent_65(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Money spent in the cloud'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_65(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart)

CPU times: user 75.9 ms, sys: 4.03 ms, total: 79.9 ms
Wall time: 72.8 ms


In [62]:
%%time
### cell 57 ###

def bar_chart_multiple_choice_multiple_selection_66(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_66(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Which of the following cloud computing platforms do you use? (Select all that apply)'
title_for_chart = 'Most popular cloud computing platforms'
models_df_2022 = grab_subset_of_data_66(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_66(models_df_2022, title_for_chart, orientation='h')

CPU times: user 158 ms, sys: 20.6 ms, total: 178 ms
Wall time: 169 ms


In [63]:
%%time
### cell 58 ###

def bar_chart_multiple_years_67(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_67(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_67(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018, question_of_interest)
        add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018, question_of_interest)
        df_2017 = grab_subset_of_data_67(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)

question_of_interest = 'Which of the following cloud computing platforms do you use? (Select all that apply)'
question_of_interest_old = 'Which of the following cloud computing platforms do you use on a regular basis?'
question_of_interest_old_2 = 'Which of the following cloud computing services have you used at work or school in the last 5 years?'
responses_df_2018.columns = responses_df_2018.columns.str.replace('Amazon Web Services (AWS)', 'Amazon Web Services (AWS) ', regex=False)
responses_df_2018.columns = responses_df_2018.columns.str.replace('Google Cloud Platform (GCP)', 'Google Cloud Platform (GCP) ', regex=False)
responses_df_2018.columns = responses_df_2018.columns.str.replace('Microsoft Azure', 'Microsoft Azure ', regex=False)
responses_df_2021.columns = responses_df_2021.columns.str.replace(question_of_interest_old, question_of_interest, regex=False)
responses_df_2020.columns = responses_df_2020.columns.str.replace(question_of_interest_old, question_of_interest, regex=False)
responses_df_2019.columns = responses_df_2019.columns.str.replace(question_of_interest_old, question_of_interest, regex=False)
responses_df_2018.columns = responses_df_2018.columns.str.replace(question_of_interest_old_2, question_of_interest, regex=False)
(cloud_df_combined, cloud_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(question_of_interest)
cloud_df_combined_percentages = convert_df_of_counts_to_percentages_67(cloud_df_combined, cloud_df_combined_counts)
cloud_df_combined_percentages = cloud_df_combined_percentages.loc[:, ['year', 'Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure ']]
title_for_chart = 'Most popular cloud computing platforms 2018-2022'
title_for_y_axis = '% of respondents'
df = cloud_df_combined_percentages.melt(id_vars=['year'], value_vars=['Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure '])
df = df.rename(columns={'variable': ''})
bar_chart_multiple_years_67(df, '', 'value', title_for_chart, title_for_y_axis)

CPU times: user 1.43 s, sys: 341 ms, total: 1.77 s
Wall time: 1.72 s


In [64]:
%%time
### cell 59 ###

def bar_chart_multiple_choice_68(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_68(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Of the cloud platforms that you are familiar with, which has the best developer experience (most enjoyable to use)? - Selected Choice'
responses_in_order = [' Amazon Web Services (AWS) ', ' Google Cloud Platform (GCP) ', ' Microsoft Azure ', ' IBM Cloud / Red Hat ', ' Oracle Cloud ', ' VMware Cloud ', ' SAP Cloud ', ' Tencent Cloud ', ' Alibaba Cloud ', ' Huawei Cloud ']
percentages = count_then_return_percent_68(responses_df_2022, question_of_interest).sort_index()[responses_in_order].iloc[::-1]
title_for_chart = 'Best developer experience according to users of multiple cloud services'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_68(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=10)

CPU times: user 76.9 ms, sys: 4.6 ms, total: 81.5 ms
Wall time: 74.9 ms


In [65]:
%%time
### cell 60 ###

def bar_chart_multiple_choice_multiple_selection_69(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_69(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following cloud computing products?'
title_for_chart = 'Most popular compute products for AWS GCP and Azure customers'
compute_df_2022 = grab_subset_of_data_69(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_69(compute_df_2022, title_for_chart, orientation='h')

CPU times: user 158 ms, sys: 16.2 ms, total: 174 ms
Wall time: 167 ms


In [66]:
%%time
### cell 61 ###

def bar_chart_multiple_choice_multiple_selection_70(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_70(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following data storage products?'
responses_df_2022.columns = responses_df_2022.columns.str.replace('(AWS/GCP/Azure customers only)', 'for AWS GCP and Azure customers')
title_for_chart = 'Most popular data storage products for AWS GCP and Azure customers'
storage_df_2022 = grab_subset_of_data_70(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_70(storage_df_2022, title_for_chart, orientation='h')

CPU times: user 222 ms, sys: 17.3 ms, total: 240 ms
Wall time: 229 ms


In [67]:
%%time
### cell 62 ###

def bar_chart_multiple_choice_multiple_selection_71(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_71(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following data products (relational databases, data warehouses, data lakes, or similar)?'
title_for_chart = 'Most popular data storage products (relational databases, data lakes, and similar)'
big_data_df_2022 = grab_subset_of_data_71(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_71(big_data_df_2022, title_for_chart, orientation='h')

CPU times: user 167 ms, sys: 19.9 ms, total: 187 ms
Wall time: 180 ms


In [68]:
%%time
### cell 63 ###

def bar_chart_multiple_choice_multiple_selection_72(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_72(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following business intelligence tools?'
title_for_chart = 'Most popular business intelligence tools'
bi_df_2022 = grab_subset_of_data_72(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_72(bi_df_2022, title_for_chart, orientation='h')

CPU times: user 159 ms, sys: 25 ms, total: 184 ms
Wall time: 176 ms


In [69]:
%%time
### cell 64 ###

def bar_chart_multiple_choice_multiple_selection_73(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_73(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following managed machine learning products on a regular basis?'
title_for_chart = 'Most popular fully managed ML services'
managed_ml_df_2022 = grab_subset_of_data_73(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_73(managed_ml_df_2022, title_for_chart, orientation='h')

CPU times: user 160 ms, sys: 20.5 ms, total: 180 ms
Wall time: 173 ms


In [70]:
%%time
### cell 65 ###

def bar_chart_multiple_choice_multiple_selection_74(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_74(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following automated machine learning tools?'
title_for_chart = 'Most popular automated machine learning products'
automl_df_2022 = grab_subset_of_data_74(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_74(automl_df_2022, title_for_chart, orientation='h')

CPU times: user 152 ms, sys: 24.7 ms, total: 177 ms
Wall time: 171 ms


In [71]:
%%time
### cell 66 ###

def bar_chart_multiple_choice_multiple_selection_75(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_75(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following products to serve your machine learning models?'
title_for_chart = 'Most popular ML model serving tools'
serving_df_2022 = grab_subset_of_data_75(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_75(serving_df_2022, title_for_chart, orientation='h')

CPU times: user 154 ms, sys: 28.9 ms, total: 183 ms
Wall time: 176 ms


In [72]:
%%time
### cell 67 ###

def bar_chart_multiple_choice_multiple_selection_76(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_76(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any tools to help monitor your machine learning models and/or experiments?'
title_for_chart = 'Most popular ML experiment tracking tools'
tracking_df_2022 = grab_subset_of_data_76(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_76(tracking_df_2022, title_for_chart, orientation='h')

CPU times: user 160 ms, sys: 32.7 ms, total: 193 ms
Wall time: 186 ms


In [73]:
%%time
### cell 68 ###

def bar_chart_multiple_choice_multiple_selection_77(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_77(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following responsible or ethical AI products in your machine learning practices?'
title_for_chart = 'Most popular tools for responsible ML'
responsible_df_2022 = grab_subset_of_data_77(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_77(responsible_df_2022, title_for_chart, orientation='h')

CPU times: user 146 ms, sys: 32.4 ms, total: 178 ms
Wall time: 172 ms


In [74]:
%%time
### cell 69 ###

def bar_chart_multiple_choice_multiple_selection_78(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_78(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Do you use any of the following types of specialized hardware when training machine learning models?'
title_for_chart = 'Most popular ML accelerators'
hardware_df_2022 = grab_subset_of_data_78(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_78(hardware_df_2022, title_for_chart, orientation='h')

CPU times: user 165 ms, sys: 12.1 ms, total: 177 ms
Wall time: 170 ms


In [75]:
%%time
### cell 70 ###

def grab_subset_of_data_79(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest_2022 = 'Do you use any of the following types of specialized hardware when training machine learning models?'
question_of_interest_2019 = 'Which types of specialized hardware do you use on a regular basis?'
responses_df_2019.columns = responses_df_2019.columns.str.replace(question_of_interest_2019, question_of_interest_2022, regex=False)
question_of_interest_2020 = 'Which types of specialized hardware do you use on a regular basis?'
responses_df_2020.columns = responses_df_2020.columns.str.replace(question_of_interest_2020, question_of_interest_2022, regex=False)
question_of_interest_2021 = 'Which types of specialized hardware do you use on a regular basis?'
responses_df_2021.columns = responses_df_2021.columns.str.replace(question_of_interest_2021, question_of_interest_2022, regex=False)
question_of_interest = question_of_interest_2022
title_for_chart = 'Most popular ML accelerators'
hardware_df_2019 = grab_subset_of_data_79(responses_df_2019, question_of_interest)
counts_2019 = hardware_df_2019[hardware_df_2019.columns[:]].count().sort_values(ascending=True)
percentages_2019 = counts_2019 * 100 / len(hardware_df_2019)
percentages_2019 = percentages_2019[['TPUs']]
hardware_df_2020 = grab_subset_of_data_79(responses_df_2020, question_of_interest)
counts_2020 = hardware_df_2020[hardware_df_2020.columns[:]].count().sort_values(ascending=True)
percentages_2020 = counts_2020 * 100 / len(hardware_df_2020)
percentages_2020 = percentages_2020[['TPUs']]
hardware_df_2021 = grab_subset_of_data_79(responses_df_2021, question_of_interest)
counts_2021 = hardware_df_2021[hardware_df_2021.columns[:]].count().sort_values(ascending=True)
percentages_2021 = counts_2021 * 100 / len(hardware_df_2021)
percentages_2021 = percentages_2021.rename({'Google Cloud TPUs ': 'TPUs', 'NVIDIA GPUs ': 'GPUs'})
percentages_2021 = percentages_2021[['TPUs']]
hardware_df_2022 = grab_subset_of_data_79(responses_df_2022, question_of_interest)
counts_2022 = hardware_df_2022[hardware_df_2022.columns[:]].count().sort_values(ascending=True)
percentages_2022 = counts_2022 * 100 / len(hardware_df_2022)
percentages_2022 = percentages_2022.rename({'TPUs ': 'TPUs', 'GPUs ': 'GPUs'})
percentages_2022 = percentages_2022[['TPUs']]
title_for_chart = 'Usage of Tensor Processing Units (TPUs) by Kagglers (2019-2022)'
title_for_y_axis = '% of respondents'
percentages_2019.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2019_' + title_for_chart + '.csv', index=True)
percentages_2020.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2020_' + title_for_chart + '.csv', index=True)
percentages_2021.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2021_' + title_for_chart + '.csv', index=True)
percentages_2022.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/2022_' + title_for_chart + '.csv', index=True)

CPU times: user 875 ms, sys: 132 ms, total: 1.01 s
Wall time: 964 ms


In [76]:
%%time
### cell 71 ###

def bar_chart_multiple_choice_80(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ''
    pd.DataFrame(response_counts_series).to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)

def count_then_return_percent_80(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest = 'Approximately how many times have you used a TPU (tensor processing unit)?'
responses_in_order = ['Never', 'Once', '2-5 times', '6-25 times', 'More than 25 times']
percentages = count_then_return_percent_80(responses_df_2022, question_of_interest).sort_index()[responses_in_order]
title_for_chart = 'Popularity of Tensor Processing Units (TPUs)'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_80(response_counts=percentages, title=title_for_chart, y_axis_title=title_for_y_axis, orientation=orientation_for_chart, num_choices=5)

CPU times: user 73.4 ms, sys: 7.61 ms, total: 81 ms
Wall time: 74.7 ms


In [77]:
%%time
### cell 72 ###

def bar_chart_multiple_years_81(df, x_column, y_column, title, y_axis_title):
    df.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def count_then_return_percent_81(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages

question_of_interest_2022 = 'Approximately how many times have you used a TPU (tensor processing unit)?'
question_of_interest_2019 = 'Have you ever used a TPU (tensor processing unit)?'
responses_df_2019[question_of_interest_2019] = responses_df_2019[question_of_interest_2019].str.replace('6-24 times', '6-25 times', regex=False)
responses_df_2019[question_of_interest_2019] = responses_df_2019[question_of_interest_2019].str.replace('> 25 times', 'More than 25 times', regex=False)
responses_df_2019.columns = responses_df_2019.columns.str.replace(question_of_interest_2019, question_of_interest_2022, regex=False)
question_of_interest = question_of_interest_2022
df_2022 = pd.DataFrame(count_then_return_percent_81(responses_df_2022, question_of_interest).sort_index())
df_2021 = pd.DataFrame(count_then_return_percent_81(responses_df_2021, question_of_interest).sort_index())
df_2020 = pd.DataFrame(count_then_return_percent_81(responses_df_2020, question_of_interest).sort_index())
df_2019 = pd.DataFrame(count_then_return_percent_81(responses_df_2019, question_of_interest).sort_index())
df_2022['year'] = '2022'
df_2021['year'] = '2021'
df_2020['year'] = '2020'
df_2019['year'] = '2019'
df_combined = pd.concat([df_2019, df_2020, df_2021, df_2022])
df_combined[x_axis_title] = df_combined.index
x_axis_title = 'Frequency of TPU usage'
df_combined.columns = ['percentage', 'year', x_axis_title]
question_of_interest = question_of_interest_2022
column_of_interest = 'percentage'
title_for_chart = 'Frequency of TPU usage for Kagglers (2019-2022)'
title_for_x_axis = x_axis_title
title_for_y_axis = '% of respondents'
df_combined = df_combined.sort_values(by=['year', 'percentage'], ascending=True)
bar_chart_multiple_years_81(df_combined, title_for_x_axis, column_of_interest, title_for_chart, title_for_y_axis)

CPU times: user 398 ms, sys: 9.42 ms, total: 408 ms
Wall time: 387 ms


In [78]:
%%time
### cell 73 ###

def bar_chart_multiple_choice_multiple_selection_82(df, title, orientation='v'):
    df_new = df.copy()
    counts = df_new[df_new.columns[:]].count().sort_values(ascending=True)
    percentages = counts * 100 / len(df_new)
    percentages.index.name = ''
    percentages.to_csv('/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/' + title + '.csv', index=True)

def grab_subset_of_data_82(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

question_of_interest = 'Who/what are your favorite media sources that report on data science topics?'
title_for_chart = 'Most popular data science media sources'
media_df_2022 = grab_subset_of_data_82(responses_df_2022, question_of_interest)
bar_chart_multiple_choice_multiple_selection_82(media_df_2022, title_for_chart, orientation='h')

CPU times: user 167 ms, sys: 23.8 ms, total: 191 ms
Wall time: 184 ms


In [79]:
%%time
### cell 74 ###

print('Total Number of Responses 2017: ',responses_df_2017.shape[0])
print('Total Number of Responses 2018: ',responses_df_2018.shape[0])
print('Total Number of Responses 2019: ',responses_df_2019.shape[0])
print('Total Number of Responses 2020: ',responses_df_2020.shape[0])
print('Total Number of Responses 2021: ',responses_df_2021.shape[0])
print('Total Number of Responses 2022: ',responses_df_2022.shape[0])
print('\nNote that survey response counts are influenced by many different factors')
print('If discussing the size of the Kaggle community, consider using the daily updated data ')
print('from the Kaggle Meta Kaggle dataset instead (see chart below)')

Total Number of Responses 2017:  16716
Total Number of Responses 2018:  23859
Total Number of Responses 2019:  19717
Total Number of Responses 2020:  20036
Total Number of Responses 2021:  25973
Total Number of Responses 2022:  23997

Note that survey response counts are influenced by many different factors
If discussing the size of the Kaggle community, consider using the daily updated data 
from the Kaggle Meta Kaggle dataset instead (see chart below)
CPU times: user 141 ms, sys: 8.9 ms, total: 150 ms
Wall time: 141 ms


In [80]:
# STEFANOS: Disabled this cell because the meta-kaggle dataset was not available.
# users = pd.read_csv("../input/meta-kaggle/Users.csv")
# print('In 2022 there were more than 10 million registered users on Kaggle: ',users['Id'].nunique())

# users['RegisterDate'] = pd.to_datetime(users['RegisterDate'])
# users_in_2016 = users[users['RegisterDate'] < '09/01/2016']
# users_in_2017 = users[users['RegisterDate'] < '09/01/2017']
# users_in_2018 = users[users['RegisterDate'] < '09/01/2018']
# users_in_2019 = users[users['RegisterDate'] < '09/01/2019']
# users_in_2020 = users[users['RegisterDate'] < '09/01/2020']
# users_in_2021 = users[users['RegisterDate'] < '09/01/2021']
# users_in_2022 = users[users['RegisterDate'] < '09/01/2022']

# new_users_in_12_months_prior_survey_2017 = users_in_2017['Id'].nunique()-users_in_2016['Id'].nunique()
# new_users_in_12_months_prior_survey_2018 = users_in_2018['Id'].nunique()-users_in_2017['Id'].nunique()
# new_users_in_12_months_prior_survey_2019 = users_in_2019['Id'].nunique()-users_in_2018['Id'].nunique()
# new_users_in_12_months_prior_survey_2020 = users_in_2020['Id'].nunique()-users_in_2019['Id'].nunique()
# new_users_in_12_months_prior_survey_2021 = users_in_2021['Id'].nunique()-users_in_2020['Id'].nunique()
# new_users_in_12_months_prior_survey_2022 = users_in_2022['Id'].nunique()-users_in_2021['Id'].nunique()

# df = pd.DataFrame({'New users in the 12 months prior to September': [new_users_in_12_months_prior_survey_2017, 
#                                  new_users_in_12_months_prior_survey_2018, 
#                                  new_users_in_12_months_prior_survey_2019, 
#                                  new_users_in_12_months_prior_survey_2020, 
#                                  new_users_in_12_months_prior_survey_2021,
#                                  new_users_in_12_months_prior_survey_2022],
#                  'Survey Year': ['2017','2018','2019','2020','2021','2022']})

In [81]:
# STEFANOS: Disable plotting
# fig = px.line(df, x="Survey Year", y="New users in the 12 months prior to September", title='Yearly Kaggle account registrations 2017-2022')
# fig.show()

In [82]:
%%time
### cell 75 ###

# Plot a histogram of user response times 
responses_only_duration = responses_df_2022['Duration (in seconds)']
responses_only_duration = pd.DataFrame(pd.to_numeric(responses_only_duration, errors='coerce')/60)
responses_only_duration.columns = ['Duration (in minutes)']
# STEFANOS: Disable plotting
# sns.displot(responses_only_duration,bins=15000).set(xlim=(0, 60))
median = round(responses_only_duration['Duration (in minutes)'].median(),0)
print('The median response time was approximately',median,'minutes.')
responses_only_duration_longer_5m = responses_only_duration[responses_only_duration['Duration (in minutes)'] > 5]  
print('The total number of respondents that took more than 5 minutes was',responses_only_duration_longer_5m.shape[0],'out of',responses_df_2022.shape[0])


The median response time was approximately 7.0 minutes.
The total number of respondents that took more than 5 minutes was 16324 out of 23997
CPU times: user 79.8 ms, sys: 4.43 ms, total: 84.2 ms
Wall time: 78.1 ms


In [83]:
end_time = time.time()
print(end_time - start_time)

36.761624336242676


**Future directions to consider:**
* Divide the population into interesting subgroups and identify interesting insights.

 * Do students have different preferences as compared to professionals?
 * Do GCP customers have different preferences as compared to AWS customers?
 * Which cloud computing platforms have seen the most growth in recent years?
 * Do salaries scale according to experience levels?  What traits might predict having a very high salary?



**Credits / Attribution:**

* The idea to use pattern matching to identify which columns are associated with which questions came from @siddhantsadangi's notebook ["Your Country VS The World"](https://www.kaggle.com/siddhantsadangi/your-country-vs-the-world-24-factors-wip) (see function grab_subset_of_data() for more detail).
* Most plotting functions were adapted from examples in the [plotly documentation](https://plotly.com/python/plotly-express/).
* This notebook (and every other public notebook on Kaggle) was released under an [Apache 2.0 license](https://www.apache.org/licenses/LICENSE-2.0). Consider clicking on the "copy & edit" button in the top right corner and then you can focus on some subset of the community that you find to be interesting!